# 07 — services gap

**What this notebook does.** Adds three more `source` layers -- `school`, `medical`, `emergency` -- to the village-level distance picture that `06_village_gap.ipynb` already builds for towers, NBN, MBSP, the coverage guide, and RICT. Same pattern nb 06 already uses for every other layer: for each of the 792 BushTel villages, straight-line (haversine) distance to the nearest site of each type, plus a count of sites within `BUFFER_KM` (10 km, the same constant nb 03/06 use).

**What it reads:**

| dataframe | file |
|---|---|
| `village_gap_df` | `data/new_processed/village_gap.csv` (nb 06 output, 792 rows) |
| `services_df` | `services/services_sites_combined.csv` (school/medical/emergency, 606 rows) |

**Where `services_sites_combined.csv` came from:** `medical` and `emergency` are Geoscience Australia / NHSD facility extracts with their own real coordinates (`medical_facilities_NT.csv`, `emergency_facilities_NORTHERN_TERRITORY.csv`). `school` is the NT Dept of Education public school list, which ships with **no coordinates at all** -- those were geocoded separately (`services/geocode_schools.py`) at locality precision against BushTel/medical/emergency suburb centroids, plus a handful of Darwin-suburb coordinates looked up individually and cited by source. 264 of 273 schools (96.7%) have a coordinate; 9 are genuinely unresolved (4 outstation schools whose own address field reads `tba`, and 5 suburbs with no match anywhere checked) and are excluded from the distance calcs below rather than guessed.

**Distances here are haversine (great-circle) on WGS84** -- consistent with how this project's distance work started (`PROJECT_CONTEXT.md` §4: "haversine now; move to EPSG:3577"). nb 03/06 have since moved the tower/MBSP distance calcs to a projected CRS (`EPSG:3577`) for better accuracy at NT's latitude/scale; this notebook doesn't, so treat `km_nearest_*` here as very slightly less precise than the equivalent tower/MBSP columns -- still a triage number, not a coverage claim, same as everywhere else in `village_gap.columns.md`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

VG_PATH = Path("data/new_processed/village_gap.csv")
SERVICES_PATH = Path("services/services_sites_combined.csv")
OUT_DIR = Path("data/new_processed"); OUT_DIR.mkdir(parents=True, exist_ok=True)

BUFFER_KM = 10  # same constant nb03/06 use for the *_10km neighbourhood counts

## 1. Load the two inputs

In [1]:
village_gap_df = pd.read_csv(VG_PATH, dtype={"sa1_code": str})
assert village_gap_df["community_id"].is_unique and len(village_gap_df) == 792
print(f"village_gap.csv: {len(village_gap_df)} rows, community_id unique")

services_df = pd.read_csv(SERVICES_PATH)
services_df = services_df.dropna(subset=["latitude", "longitude"]).copy()
print(f"services_sites_combined.csv: {len(services_df)} geocoded rows "
      f"({dict(services_df['source'].value_counts())})")

village_gap.csv: 792 rows, community_id unique
services_sites_combined.csv: 597 geocoded rows ({'school': 264, 'medical': 181, 'emergency': 152})

## 2. Nearest site + 10km buffer count, per source

Same haversine-argmin loop nb 05/06 already use for towers and MBSP, just run once per source (`school` / `medical` / `emergency`) against every village.

In [1]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

BUFFER_COL = {"school": "n_schools_10km", "medical": "n_medical_10km", "emergency": "n_emergency_10km"}
village_lat = village_gap_df["latitude"].to_numpy()
village_lon = village_gap_df["longitude"].to_numpy()

results = {}
for source, prefix in [("school", "school"), ("medical", "medical"), ("emergency", "emergency")]:
    sub = services_df[services_df["source"] == source].reset_index(drop=True)
    s_lat, s_lon = sub["latitude"].to_numpy(), sub["longitude"].to_numpy()

    nearest_km = np.empty(len(village_gap_df))
    nearest_name = np.empty(len(village_gap_df), dtype=object)
    nearest_type = np.empty(len(village_gap_df), dtype=object)
    n_within_buffer = np.empty(len(village_gap_df), dtype=int)

    for i in range(len(village_gap_df)):
        d = haversine_km(village_lat[i], village_lon[i], s_lat, s_lon)
        j = np.argmin(d)
        nearest_km[i] = round(float(d[j]), 2)
        nearest_name[i] = sub["name"].iloc[j]
        nearest_type[i] = sub["type"].iloc[j] if "type" in sub.columns else None
        n_within_buffer[i] = int((d <= BUFFER_KM).sum())

    results[f"km_nearest_{prefix}"] = nearest_km
    results[f"nearest_{prefix}_name"] = nearest_name
    results[f"nearest_{prefix}_type"] = nearest_type
    results[BUFFER_COL[source]] = n_within_buffer

    print(f"{source}: n={len(sub)}  km_nearest_{prefix} median={np.median(nearest_km):.2f} "
          f"p90={np.percentile(nearest_km, 90):.2f} max={nearest_km.max():.2f}  "
          f"villages with 0 within {BUFFER_KM}km: {int((n_within_buffer==0).sum())}")

school: n=264  km_nearest_school median=26.80 p90=80.10 max=244.78  villages with 0 within 10km: 565
medical: n=181  km_nearest_medical median=30.18 p90=100.38 max=286.94  villages with 0 within 10km: 588
emergency: n=152  km_nearest_emergency median=30.93 p90=88.43 max=244.89  villages with 0 within 10km: 600

### Validate

Every village must get a value for every source (services files have no missing coordinates left after the `dropna` above), and every village coordinate must stay inside the NT bounding box nb 05 already asserts on.

In [1]:
village_gap_services_df = village_gap_df.copy()
for col, arr in results.items():
    village_gap_services_df[col] = arr

assert village_gap_services_df["km_nearest_school"].notna().all()
assert village_gap_services_df["km_nearest_medical"].notna().all()
assert village_gap_services_df["km_nearest_emergency"].notna().all()
assert village_gap_services_df["latitude"].between(-26.5, -10.0).all()
assert village_gap_services_df["longitude"].between(128.5, 138.5).all()
print("asserts passed: no missing nearest-service values; village coords in NT box")

asserts passed: no missing nearest-service values; village coords in NT box

## 3. Does adding services change the picture nb 06 already drew?

`distance_gap_candidate` (nb 06) is a **mobile-tower** gap flag. Checking it against the new services columns is a sanity read, not a new combined flag -- deciding whether/how to combine "no mobile" with "no school/clinic/police nearby" into one tier is a judgement call for the report, not something to bake into the pipeline silently.

In [1]:
gap = village_gap_services_df[village_gap_services_df["distance_gap_candidate"] == True]
print(f"{len(gap)} villages are distance_gap_candidate (mobile tower gap, from nb06)")
print(f"  of those, median km_nearest_school = {gap['km_nearest_school'].median():.1f}, "
      f"median km_nearest_medical = {gap['km_nearest_medical'].median():.1f}, "
      f"median km_nearest_emergency = {gap['km_nearest_emergency'].median():.1f}")

all_three_close = ((village_gap_services_df["n_schools_10km"] > 0) &
                    (village_gap_services_df["n_medical_10km"] > 0) &
                    (village_gap_services_df["n_emergency_10km"] > 0))
print(f"  villages with a school AND medical AND emergency site all within 10km: "
      f"{int(all_three_close.sum())} / 792")

384 villages are distance_gap_candidate (mobile tower gap, from nb06)
  of those, median km_nearest_school = 42.7, median km_nearest_medical = 46.2, median km_nearest_emergency = 47.7
  villages with a school AND medical AND emergency site all within 10km: 161 / 792

## 4. Write outputs

Two files: the full `village_gap.csv` with the 12 new columns appended (`village_gap_with_services.csv` -- written alongside, not overwriting nb 06's own output), and a compact `community_id`-keyed table of just the new columns (`village_services_gap.csv`) for anyone who wants to join it onto something else.

In [1]:
out_csv = OUT_DIR / "village_gap_with_services.csv"
village_gap_services_df.to_csv(out_csv, index=False)
print(f"wrote {out_csv}  ({village_gap_services_df.shape[0]} rows x {village_gap_services_df.shape[1]} cols)")

compact_cols = ["community_id", "community_name"] + list(results.keys())
compact_csv = OUT_DIR / "village_services_gap.csv"
village_gap_services_df[compact_cols].to_csv(compact_csv, index=False)
print(f"wrote {compact_csv}  ({len(village_gap_services_df)} rows x {len(compact_cols)} cols)")

wrote data/new_processed/village_gap_with_services.csv  (792 rows x 43 cols)
wrote data/new_processed/village_services_gap.csv  (792 rows x 14 cols)

## 5. Known limitations (be upfront about these in the report)

- **9 schools have no coordinate** (4 outstation schools with a literal `tba` address, 5 more with no match found anywhere checked) and are silently excluded from every `*_school*` calc above -- they don't bias the numbers wrong, they're just absent.
- **Haversine, not EPSG:3577** -- unlike nb 03/06's tower/MBSP distances, these three columns are plain great-circle distance on WGS84. Close enough for triage; worth aligning with the rest of the pipeline's projected-CRS method if precision starts to matter.
- **`nearest_*_type` is a mixed bag per source** -- for `school` it's stage offered (Primary/Middle/...), for `medical` it's `NHSD_SERVICE_TYPE`, for `emergency` it's the facility class (Policing/SES/...). Don't compare this column across sources.
- **No population weighting** -- exactly like every other `km_*` column in this pipeline, per §2 of `PROJECT_CONTEXT.md`: population lives on `sa1_report.csv`, joined at SA1 level, never inferred from a village-level distance table.